# 01. Голова эмбеддингов поверх замороженного энкодера

Запускается дважды: `ENCODER = "e5"` и `ENCODER = "bge"` (настройки энкодеров в `config.ENCODERS`).

Схема:
1. Бэкбон (multilingual-e5-base / USER-bge-m3) заморожен. Его эмбеддинги для всех запросов
   и объявлений train считаются один раз и сохраняются в `.npy` (`artifacts/<enc>/raw`).
2. Поверх кэша обучается голова `DoubleGeGLUHead` (два GeGLU-блока с residual):
   InfoNCE с in-batch негативами, как `MultipleNegativesRankingLoss` (scale 20).
3. После каждой эпохи считается recall@50 на отложенных 10% запросов; сохраняется лучшая эпоха
   (`artifacts/<enc>/head`).

Полный файнтюн e5-base на 6 ГБ VRAM упирался в память. С замороженным бэкбоном эпоха идёт
минуты, дорого только однократное кодирование 345к объявлений (около часа).

Отложенные запросы (`BIENCODER_VAL_FRACTION`, seed 42) - те же, на которых потом учится
ре-ранкер в 03: для запросов, которые голова видела, косинус завышен.

In [ ]:
import os
import random
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

import config as C
from custom_head import DoubleGeGLUHead
from data import load_train, biencoder_val_uids
from text_prep import build_embedding_query_text, build_item_text_for_embedding

In [ ]:
ENCODER = "e5"          # "e5" или "bge"

cfg = C.ENCODERS[ENCODER]
paths = C.enc_paths(ENCODER)

BATCH_SIZE = 256        # батч головы по готовым векторам
EPOCHS = 10
HEAD_LR = 2e-4
WARMUP_RATIO = 0.1
SCALE = 20.0            # как в MultipleNegativesRankingLoss
SEED = 42
Q_BATCH = 128           # батч кодирования запросов

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Данные, тексты, разбиение

In [ ]:
tq, ti, tp = load_train()
print(f"queries={len(tq)} items={len(ti)} pairs={len(tp)}")

q_text = cfg["q_prefix"] + build_embedding_query_text(tq)
it_text = cfg["p_prefix"] + build_item_text_for_embedding(ti)
print(it_text.iloc[0][:300])

# разбиение по query_uid - то же, что data.biencoder_val_uids в 03
val_uids = set(biencoder_val_uids(tq, seed=SEED, frac=C.BIENCODER_VAL_FRACTION))
train_pairs = tp[~tp["query_uid"].isin(val_uids)]
val_pairs = tp[tp["query_uid"].isin(val_uids)]
print(f"train pairs: {len(train_pairs)}, val pairs: {len(val_pairs)}")

# позиции строк tq/ti = позиции в кэше эмбеддингов
uid_to_pos = {u: i for i, u in enumerate(tq["query_uid"].values)}
item_to_pos = {i: k for k, i in enumerate(ti["item_id"].values)}
train_q_pos = train_pairs["query_uid"].map(uid_to_pos).to_numpy(np.int64)
train_i_pos = train_pairs["item_id"].map(item_to_pos).to_numpy(np.int64)
val_i_pos = val_pairs["item_id"].map(item_to_pos).to_numpy(np.int64)

## Замороженный бэкбон и кэш его эмбеддингов

In [ ]:
base = SentenceTransformer(cfg["base_model"], device=DEVICE)
if cfg["max_seq_length"]:
    base.max_seq_length = cfg["max_seq_length"]
for p in base[0].auto_model.parameters():
    p.requires_grad = False
dim = base[1].get_sentence_embedding_dimension()
head = DoubleGeGLUHead(dim=dim).to(DEVICE)

os.makedirs(paths["raw"], exist_ok=True)
q_path = os.path.join(paths["raw"], "query_emb.npy")
i_path = os.path.join(paths["raw"], "item_emb.npy")
if not (os.path.exists(q_path) and os.path.exists(i_path)):
    backbone = SentenceTransformer(modules=[base[0], base[1]], device=DEVICE).half()   # только forward
    np.save(q_path, backbone.encode(q_text.tolist(), batch_size=Q_BATCH, show_progress_bar=True,
                                    convert_to_numpy=True).astype(np.float32))
    np.save(i_path, backbone.encode(it_text.tolist(), batch_size=cfg["raw_batch"], show_progress_bar=True,
                                    convert_to_numpy=True).astype(np.float32))
    del backbone
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# mmap: в RAM не читается целиком
q_cache = np.load(q_path, mmap_mode="r")
i_cache = np.load(i_path, mmap_mode="r")
print(q_cache.shape, i_cache.shape)

## Оценка: recall@50 на отложенных запросах

Корпус для оценки - объявления из отложенных пар и 20 000 случайных объявлений train.
Это грубый прокси, нужен только для выбора эпохи.

In [ ]:
rng_eval = np.random.default_rng(SEED)
rel_pos = np.unique(val_i_pos)
distractors = rng_eval.choice(np.setdiff1d(np.arange(len(ti)), rel_pos),
                              size=min(20000, len(ti) - len(rel_pos)), replace=False)
eval_corpus = np.unique(np.concatenate([rel_pos, distractors]))
val_query_pos = np.array([uid_to_pos[u] for u in val_pairs["query_uid"].unique()], dtype=np.int64)
relevant = {}
for uid, ipos in zip(val_pairs["query_uid"], val_i_pos):
    relevant.setdefault(uid, set()).add(int(ipos))
tq_uid = tq["query_uid"].values


@torch.no_grad()
def embed_cached(cache, idx, bs=512):
    out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.asarray(cache[idx[s:s + bs]], dtype=np.float32)).to(DEVICE)
        out.append(F.normalize(head({"sentence_embedding": x})["sentence_embedding"], dim=-1).cpu())
    return torch.cat(out)


@torch.no_grad()
def recall_at_k(k=50, bs=512):
    head.eval()
    corpus = embed_cached(i_cache, eval_corpus)
    recalls = []
    for s in range(0, len(val_query_pos), bs):
        idx = val_query_pos[s:s + bs]
        top = (embed_cached(q_cache, idx) @ corpus.T).topk(min(k, len(eval_corpus)), dim=-1).indices.numpy()
        for row, qpos in enumerate(idx):
            rel = relevant.get(tq_uid[qpos])
            if rel:
                recalls.append(len(set(eval_corpus[top[row]].tolist()) & rel) / len(rel))
    head.train()
    return float(np.mean(recalls))


print("recall@50 до обучения:", recall_at_k())

## Обучение головы

In [ ]:
os.makedirs(paths["checkpoints"], exist_ok=True)
n_train = len(train_q_pos)
steps = (n_train + BATCH_SIZE - 1) // BATCH_SIZE
optimizer = AdamW(head.parameters(), lr=HEAD_LR)
scheduler = get_linear_schedule_with_warmup(optimizer, int(steps * EPOCHS * WARMUP_RATIO), steps * EPOCHS)

history, best_epoch, best_recall = [], -1, -1.0
for epoch in range(EPOCHS):
    head.train()
    perm = np.random.default_rng(SEED + epoch).permutation(n_train)
    total = 0.0
    for step in tqdm(range(steps), desc=f"epoch {epoch + 1}/{EPOCHS}"):
        b = perm[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
        qx = torch.from_numpy(np.asarray(q_cache[train_q_pos[b]], dtype=np.float32)).to(DEVICE)
        ix = torch.from_numpy(np.asarray(i_cache[train_i_pos[b]], dtype=np.float32)).to(DEVICE)
        qo = F.normalize(head({"sentence_embedding": qx})["sentence_embedding"], dim=-1)
        io = F.normalize(head({"sentence_embedding": ix})["sentence_embedding"], dim=-1)
        sims = (qo @ io.T) * SCALE                         # остальные объявления батча - негативы
        loss = F.cross_entropy(sims, torch.arange(len(b), device=DEVICE))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total += loss.item()
    rec = recall_at_k()
    history.append({"epoch": epoch + 1, "loss": total / steps, "recall@50": rec})
    print(f"epoch {epoch + 1}: loss={total / steps:.4f} val recall@50={rec:.4f}")
    head.save(os.path.join(paths["checkpoints"], f"epoch_{epoch + 1}"))
    if rec > best_recall:
        best_recall, best_epoch = rec, epoch + 1

print(f"лучшая эпоха {best_epoch}, recall@50={best_recall:.4f}")
pd.DataFrame(history)

## Лучшая голова -> `artifacts/<enc>/head`

In [ ]:
if os.path.exists(paths["head"]):
    shutil.rmtree(paths["head"])
shutil.copytree(os.path.join(paths["checkpoints"], f"epoch_{best_epoch}"), paths["head"])
print("сохранено:", paths["head"], "- дальше: python 02_encode.py", ENCODER)